# Pooled current-sheet thickness across the electron-beta scan

The same analysis as `PooledThicknessProtonBeta`, with the proton beta held fixed and the electron beta scanned.
`pta.sheets.pooled_thickness` does the work; this notebook only says which runs, snapshots and filter.

In [ ]:
import csv

import matplotlib.pyplot as plt

from pta import config
from pta.plotting import use_style
from pta.sheets import pooled_thickness

use_style()
RESULTS = config.results_dir("cs_identification", "PooledThicknessElectronBeta")

In [ ]:
BETA_P = 1                            # proton beta, held fixed
BETAS_E = [0.0625, 1, 16]             # electron beta values to scan
BRMS = 0.24
TIMES = range(200, 250, 5)

K_CUTOFF = 14.12 / 3                  # the beta_p = 1 cutoff, the same for every electron beta
XI = 1.6

METHODS = ("m1", "m2", "m3")
N_BOOTSTRAP = 1000

In [ ]:
summary = []
for beta_e in BETAS_E:
    pooled = pooled_thickness(beta_p=BETA_P, beta_e=beta_e, brms=BRMS, times=TIMES,
                              k=K_CUTOFF, Xi=XI, method=METHODS, bootstrap=N_BOOTSTRAP)
    folder = config.results_dir("cs_identification", "PooledThicknessElectronBeta", f"beta_e{beta_e}")
    figure_path, array_path = pooled.save(folder, f"pooled_thickness_beta_e{beta_e}")
    print(f"beta_e = {beta_e}: {pooled.sheets_per_time}\n  -> {figure_path}")
    plt.show()
    summary += [dict(run=f"beta_e{beta_e}", **row) for row in pooled.statistics]

In [ ]:
# One row per run and method
columns = ["run", "method", "n_snapshots", "n", "median_lognorm", "sigma_lognorm", "median_se", "ci_lo", "ci_hi",
           "mean", "median_emp"]
with open(RESULTS / "pooled_summary.csv", "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(summary)
print(f"saved {RESULTS / 'pooled_summary.csv'}")